In [ ]:
#====================================================================================================#
#                                                                                                    #
#                                                        ██╗   ██╗   ████████╗ █████╗ ██████╗        #
#      AEC1 - BAIN                                       ██║   ██║   ╚══██╔══╝██╔══██╗██╔══██╗       #
#                                                        ██║   ██║█████╗██║   ███████║██║  ██║       #
#      created:        19/03/2026  -  03:00:51           ██║   ██║╚════╝██║   ██╔══██║██║  ██║       #
#      last change:    22/03/2026  -  11:34:43           ╚██████╔╝      ██║   ██║  ██║██████╔╝       #
#                                                         ╚═════╝       ╚═╝   ╚═╝  ╚═╝╚═════╝        #
#                                                                                                    #
#      Ismael Hernandez Clemente                         ismael.hernandez@live.u-tad.com             #
#                                                                                                    #
#      Github:                                           https://github.com/ismaelucky342            #
#                                                                                                    #
#====================================================================================================# 

# AEC1 – Analisis de Sentimiento y Tendencias en Redes Sociales

In [ ]:
import re
import json
import unicodedata
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from wordcloud import WordCloud

print("Libraries loaded successfully.")

---
## Clase `DataExtractor`

Esta clase es mi punto central: aqui cargo, limpio, analizo y visualizo.

In [ ]:
class DataExtractor:
    """
    Clase principal donde centralizo la extraccion, limpieza y analisis de tweets.

    Atributos

    source_file : str
        Ruta al archivo de datos (CSV o JSON).
    data : pd.DataFrame or None
        DataFrame con los datos cargados.
    chunksize : int
        Numero de registros por chunk al leer CSVs grandes.
    """

    # Compilo los patrones una sola vez para no repetir trabajo en cada llamada.
    _RE_URL      = re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE)
    _RE_MENTION  = re.compile(r'@\w+')
    _RE_HASHTAG  = re.compile(r'#(\w+)')
    # Conservo letras, digitos, espacios y el simbolo de hashtag.
    _RE_SPECIAL  = re.compile(r'[^\w\s#]', re.UNICODE)
    _RE_SPACES   = re.compile(r'\s+')

    def __init__(self, source_file: str, chunksize: int = 100_000):
        """
        Inicializo el extractor con el archivo de origen.

        Parameters
        ----------
        source_file : str
            Ruta al archivo de datos (CSV o JSON).
        chunksize : int, optional
            Tamano del chunk para lectura de grandes archivos (default 100 000).
        """
        self.source_file = source_file
        self.data: pd.DataFrame | None = None
        self.chunksize = chunksize

    def load_data(self) -> pd.DataFrame:
        """
        Cargo los datos del archivo de origen.

        Detecto automaticamente si el archivo es CSV o JSON.
        Para CSVs grandes uso `chunksize` y luego concateno los chunks.
        Desactivo `low_memory` para evitar inferencias raras de tipos.
        El fin de linea se fija a '\n' (LF) para compatibilidad con Kaggle.

        """
        path = Path(self.source_file)
        ext  = path.suffix.lower()

        if ext == '.json':
            self.data = pd.read_json(self.source_file, encoding='utf-8')

        elif ext == '.csv':
            # Leo en chunks para no cargar todo el CSV de golpe.
            chunks = pd.read_csv(
                self.source_file,
                encoding='utf-8',
                low_memory=False,
                chunksize=self.chunksize,
                lineterminator='\n',  # Dejo LF para evitar problemas con finales de linea.
            )
            self.data = pd.concat(chunks, ignore_index=True)

        else:
            raise ValueError(f"Unsupported file format: '{ext}'. Use CSV or JSON.")

        # Si el CSV viene con CRLF, limpio el '\r' de los nombres de columna.
        self.data.columns = [c.strip().replace('\r', '') for c in self.data.columns]

        print(f"[load_data] Loaded {len(self.data):,} rows – columns: {list(self.data.columns)}")
        return self.data

    def clean_text(self, text: str) -> str:
        """
        Limpio y normalizo un tweet.

        Pasos
        1. Convierto a string y elimino espacios al inicio/fin.
        2. Paso todo a minusculas.
        3. Elimino URLs (http/https/www).
        4. Quito menciones (@usuario), no me aportan para hashtags.
        5. Quito emojis y simbolos con categorias Unicode So/Sk/Cs.
        6. Elimino caracteres especiales salvo letras, digitos, espacios y '#'.
        7. Colapso espacios redundantes.

        """
        if not isinstance(text, str):
            text = str(text) if pd.notna(text) else ''

        text = text.strip()

        # Primero bajo todo a minusculas y quito URLs y menciones, luego elimino emojis y caracteres raros, y al final colapso espacios.
        text = text.lower()
        text = self._RE_URL.sub(' ', text)
        text = self._RE_MENTION.sub(' ', text)
        text = ''.join(
            ch for ch in text
            if unicodedata.category(ch) not in ('So', 'Sk', 'Cs')
        )
        text = self._RE_SPECIAL.sub(' ', text)
        text = self._RE_SPACES.sub(' ', text).strip()

        return text

    def extract_hashtags(self, text: str) -> list:
        """
        Extraigo y devuelvo una lista de hashtags presentes en el texto.

        Uso el patron `#(\w+)` y devuelvo hashtags en minusculas, sin '#'.
        """
        if not isinstance(text, str):
            return []
        hashtags = self._RE_HASHTAG.findall(text)
        # Devuelvo hashtags en minusculas y deduplicados sin perder el orden.
        seen, result = set(), []
        for tag in hashtags:
            tag_lower = tag.lower()
            if tag_lower not in seen:
                seen.add(tag_lower)
                result.append(tag_lower)
        return result

    def analytics_hashtags_extended(self) -> dict:
        """
        Analisis avanzado de hashtags sobre self.data.

        Pasos
        1. Aplico clean_text a la columna 'text'.
        2. Extraigo hashtags por tweet -> nueva columna 'hashtags'.
        3. Convierto 'date' a datetime y me quedo con la fecha.
        4. Exploto la lista de hashtags (1 fila por hashtag).
        5. Calculo tres agregaciones:
           - **overall**: frecuencia global de cada hashtag.
           - **by_user**: frecuencia por (usuario, hashtag).
           - **by_date**: frecuencia por (fecha, hashtag).
        """
        if self.data is None:
            raise RuntimeError("Data not loaded. Call load_data() first.")

        df = self.data.copy()

        # Aqui limpio texto, saco hashtags y normalizo fechas antes de explotar y agrupar.
        print("[analytics] Cleaning text...")
        df['clean_text'] = df['text'].apply(self.clean_text)

        print("[analytics] Extracting hashtags...")
        df['hashtags'] = df['clean_text'].apply(self.extract_hashtags)

        print("[analytics] Parsing dates...")
        df['date'] = pd.to_datetime(df['date'], utc=True, errors='coerce').dt.date

        df_exp = df.explode('hashtags').dropna(subset=['hashtags'])
        df_exp = df_exp[df_exp['hashtags'].str.strip() != '']
        df_exp = df_exp.rename(columns={'hashtags': 'hashtag'})

        # Calculo la frecuencia global por hashtag.
        overall = (
            df_exp.groupby('hashtag', as_index=False)
                  .size()
                  .rename(columns={'size': 'frequency'})
                  .sort_values('frequency', ascending=False)
                  .reset_index(drop=True)
        )

        # Luego la frecuencia por usuario.
        by_user = (
            df_exp.groupby(['user_name', 'hashtag'], as_index=False)
                  .size()
                  .rename(columns={'size': 'frequency'})
                  .sort_values('frequency', ascending=False)
                  .reset_index(drop=True)
        )

        # Y finalmente la frecuencia por fecha.
        by_date = (
            df_exp.groupby(['date', 'hashtag'], as_index=False)
                  .size()
                  .rename(columns={'size': 'frequency'})
                  .sort_values(['date', 'frequency'], ascending=[True, False])
                  .reset_index(drop=True)
        )

        print(f"[analytics] Done – {len(overall)} unique hashtags found.")
        return {'overall': overall, 'by_user': by_user, 'by_date': by_date}

    def generate_hashtag_wordcloud(
        self,
        overall_df: pd.DataFrame = None,
        max_words: int = 100,
        figsize: tuple = (10, 6),
    ) -> None:
        """
        Genero y muestro una wordcloud de hashtags por frecuencia global.
        """
        if overall_df is None:
            overall_df = self.analytics_hashtags_extended()['overall']

        # Paso a dict {hashtag: frecuencia} para alimentar la nube.
        freq_dict = dict(zip(overall_df['hashtag'], overall_df['frequency']))

        # Genero la wordcloud con ese diccionario.
        wc = WordCloud(
            width=1200,
            height=700,
            max_words=max_words,
            background_color='white',
            colormap='viridis',
            prefer_horizontal=0.9,
        ).generate_from_frequencies(freq_dict)

        fig, ax = plt.subplots(figsize=figsize)
        ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        ax.set_title('Top Hashtags – Bitcoin Tweets', fontsize=16, fontweight='bold', pad=15)
        plt.tight_layout()
        plt.savefig('wordcloud_hashtags.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("[wordcloud] Saved as 'wordcloud_hashtags.png'")


print("DataExtractor class defined successfully.")

---
## 1. Carga de datos

Descargo el dataset de Kaggle y lo dejo en el mismo directorio del notebook:  
`Bitcoin_tweets_dataset_2.csv`

In [ ]:
SOURCE_FILE = 'Bitcoin_tweets_dataset_2.csv'

extractor = DataExtractor(source_file=SOURCE_FILE, chunksize=100_000)
df = extractor.load_data()
df.head(3)

In [ ]:
# Dejo un resumen rapido para ver forma, columnas y tipos.
print(f"Shape       : {df.shape}")
print(f"Columns     : {list(df.columns)}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB\n")
print(df.dtypes)

In [ ]:
# Chequeo rapido de nulos para hacerme una idea del estado.
df.isnull().sum()

---
## 2. Almacenamiento de datos limpios

Guardo el dataset en CSV con UTF-8 para asegurar compatibilidad.

> **Por que CSV:** es el formato mas comodo para datos tabulares y funciona genial con `chunksize`.

In [ ]:
OUTPUT_CSV = 'Bitcoin_tweets_cleaned.csv'

df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8', lineterminator='\n')
print(f"Dataset saved to '{OUTPUT_CSV}' ({Path(OUTPUT_CSV).stat().st_size / 1e6:.1f} MB)")

---
## 3. Limpieza y normalizacion

Aplico `clean_text` a la columna `text` y comparo antes/despues.

In [ ]:
# Cojo un tweet en bruto y veo el antes y despues de la limpieza.
sample_raw = df['text'].dropna().iloc[0]
print("RAW :", repr(sample_raw))
print("CLEAN:", repr(extractor.clean_text(sample_raw)))

In [ ]:
# Aplico la limpieza a todo el dataset y luego comparo texto original vs limpio en cinco ejemplos.
df['clean_text'] = df['text'].apply(extractor.clean_text)

pd.set_option('display.max_colwidth', 120)
df[['text', 'clean_text']].head(5)

---
## 4. Extraccion de hashtags

In [ ]:
# Hago una prueba rapida para ver que extrae el patron de hashtags.
sample_tweet = "Just bought more #Bitcoin! $BTC is going to the moon #crypto #hodl"
print(extractor.extract_hashtags(sample_tweet))

In [ ]:
df['hashtags'] = df['clean_text'].apply(extractor.extract_hashtags)

n_with_hashtags = (df['hashtags'].apply(len) > 0).sum()
print(f"Tweets with at least one hashtag: {n_with_hashtags:,} / {len(df):,} "
      f"({n_with_hashtags/len(df)*100:.1f}%)")
df[['text', 'hashtags']].head(5)

---
## 5. Analisis extendido de hashtags

In [ ]:
analytics = extractor.analytics_hashtags_extended()

overall = analytics['overall']
by_user = analytics['by_user']
by_date = analytics['by_date']

In [ ]:
# Empiezo con el top 20 global para ver las etiquetas mas repetidas.
top20 = overall.head(20)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top20['hashtag'][::-1], top20['frequency'][::-1], color='steelblue')
ax.bar_label(bars, fmt='%d', padding=3, fontsize=9)
ax.set_xlabel('Frequency')
ax.set_title('Top 20 Hashtags – Global Frequency', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('top20_hashtags.png', dpi=150, bbox_inches='tight')
plt.show()
print(top20.to_string(index=False))

In [ ]:
# Aqui miro usuarios con mas uso de hashtags para intuir posibles bots.
user_totals = (
    by_user.groupby('user_name', as_index=False)['frequency']
           .sum()
           .sort_values('frequency', ascending=False)
)

print("Top 15 usuarios por ocurrencias de hashtags (posibles bots):")
print(user_totals.head(15).to_string(index=False))

# Calculo cuanto concentra el top 1% de usuarios en uso total de hashtags.
top1pct = max(1, int(len(user_totals) * 0.01))
total_hashtag_uses = user_totals['frequency'].sum()
top1pct_uses = user_totals.head(top1pct)['frequency'].sum()
print(f"\nTop 1% of users ({top1pct:,}) account for "
      f"{top1pct_uses / total_hashtag_uses * 100:.1f}% of all hashtag uses")

In [ ]:
# Me quedo con los top hashtags y miro su evolucion diaria.
TOP_N_TAGS = 5
top_tags = overall.head(TOP_N_TAGS)['hashtag'].tolist()

pivot = (
    by_date[by_date['hashtag'].isin(top_tags)]
    .pivot_table(index='date', columns='hashtag', values='frequency', aggfunc='sum')
    .fillna(0)
    .sort_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
pivot.plot(ax=ax, linewidth=1.5, alpha=0.85)
ax.set_title(f'Daily frequency of Top {TOP_N_TAGS} Hashtags', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Frequency')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('hashtag_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Wordcloud de hashtags

In [ ]:
# Reuso el DataFrame overall para evitar recalculos.
extractor.generate_hashtag_wordcloud(overall_df=overall, max_words=100, figsize=(12, 7))

---
## 7. Wordcloud de palabras frecuentes (sin hashtags)

Analisis extra para ver el vocabulario general.

In [ ]:
from collections import Counter

# Defino stopwords basicas en ingles y español para limpiar el conteo.
STOPWORDS = {
    'the', 'a', 'an', 'is', 'in', 'it', 'of', 'to', 'and', 'for',
    'on', 'at', 'i', 'this', 'that', 'with', 'are', 'be', 'as',
    'by', 'was', 'has', 'have', 'from', 'or', 'will', 'not', 'you',
    'rt', 'amp', 'de', 'la', 'el', 'en', 'es', 'se', 'lo', 'un',
    'my', 'we', 'he', 'she', 'they', 'but', 'so', 'if', 'its',
    'via', 'all', 'up', 'no', 'do', 'get', 'can', 'now', 'just',
    'more', 'our', 'out', 'new', 'what', 'when', 'than', 'your'
}

# Cuento palabras quitando hashtags y stopwords para ver vocabulario general.
word_counts: Counter = Counter()
for tweet in df['clean_text'].dropna():
    words = [
        w for w in tweet.split()
        if not w.startswith('#')
        and w not in STOPWORDS
        and len(w) > 2
    ]
    word_counts.update(words)

wc = WordCloud(
    width=1200, height=700, max_words=100,
    background_color='white', colormap='plasma',
).generate_from_frequencies(word_counts)

fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Most Frequent Words in Bitcoin Tweets (excl. hashtags & stopwords)',
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('wordcloud_words.png', dpi=150, bbox_inches='tight')
plt.show()

print("Top 15 words:", word_counts.most_common(15))

---
## 8. (Bonus) Mini-dashboard con Streamlit

El dashboard esta en `dashboard.py` dentro de este mismo directorio.

Para lanzarlo:
```bash
streamlit run dashboard.py
```